In [1]:
from __future__ import annotations

import csv
import html
import math
import sys
from pathlib import Path
from statistics import mean

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from samples.ml_tasks.common import (  # noqa: E402
    FEATURES,
    generate_classification_rows,
    save_json,
)


ARTIFACT_DIR = Path("artifacts/agent_rounds/20260615_round2/data_validation")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def summarize(rows, feature):
    values = [float(row[feature]) for row in rows if row.get(feature) is not None]
    return {
        "count": len(values),
        "missing": len(rows) - len(values),
        "mean": mean(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }


def psi(expected, actual, bins=8):
    low = min(min(expected), min(actual))
    high = max(max(expected), max(actual))
    width = (high - low) / bins if high > low else 1.0
    score = 0.0
    for idx in range(bins):
        start = low + idx * width
        end = high if idx == bins - 1 else start + width
        exp = sum(start <= value <= end for value in expected) / len(expected)
        act = sum(start <= value <= end for value in actual) / len(actual)
        exp = max(exp, 1e-6)
        act = max(act, 1e-6)
        score += (act - exp) * math.log(act / exp)
    return score


baseline = generate_classification_rows(n=700, seed=53, drift=0.0)
production = generate_classification_rows(n=700, seed=61, drift=0.85)

# Simulate two production data-quality regressions alongside population drift.
for row in production[::37]:
    row["nps"] = None
for row in production[::53]:
    row["support_load"] = None

feature_report = {}
for feature in FEATURES:
    base_summary = summarize(baseline, feature)
    prod_summary = summarize(production, feature)
    base_values = [float(row[feature]) for row in baseline if row.get(feature) is not None]
    prod_values = [float(row[feature]) for row in production if row.get(feature) is not None]
    feature_report[feature] = {
        "baseline": base_summary,
        "production": prod_summary,
        "psi": psi(base_values, prod_values),
        "mean_shift": prod_summary["mean"] - base_summary["mean"],
    }

alerts = [
    {
        "feature": feature,
        "psi": report["psi"],
        "missing": report["production"]["missing"],
        "mean_shift": report["mean_shift"],
    }
    for feature, report in feature_report.items()
    if report["psi"] > 0.2 or report["production"]["missing"] > 0
]
alerts = sorted(alerts, key=lambda item: (item["missing"] > 0, item["psi"]), reverse=True)

json_path = save_json(
    ARTIFACT_DIR / "data_validation_drift.json",
    {"features": feature_report, "alerts": alerts},
)

csv_path = ARTIFACT_DIR / "feature_drift_summary.csv"
with csv_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "feature",
            "baseline_mean",
            "production_mean",
            "mean_shift",
            "production_missing",
            "psi",
            "alert",
        ],
    )
    writer.writeheader()
    alert_features = {alert["feature"] for alert in alerts}
    for feature, report in sorted(feature_report.items(), key=lambda item: item[1]["psi"], reverse=True):
        writer.writerow(
            {
                "feature": feature,
                "baseline_mean": f"{report['baseline']['mean']:.6f}",
                "production_mean": f"{report['production']['mean']:.6f}",
                "mean_shift": f"{report['mean_shift']:.6f}",
                "production_missing": report["production"]["missing"],
                "psi": f"{report['psi']:.6f}",
                "alert": feature in alert_features,
            }
        )


def metric(value):
    return f"{value:.3f}" if isinstance(value, float) else str(value)


alert_features = {alert["feature"] for alert in alerts}
rows = []
for feature, report in sorted(feature_report.items(), key=lambda item: item[1]["psi"], reverse=True):
    rows.append(
        "<tr>"
        f"<td>{html.escape(feature)}</td>"
        f"<td>{metric(report['baseline']['mean'])}</td>"
        f"<td>{metric(report['production']['mean'])}</td>"
        f"<td>{metric(report['mean_shift'])}</td>"
        f"<td>{report['production']['missing']}</td>"
        f"<td>{metric(report['psi'])}</td>"
        f"<td>{'yes' if feature in alert_features else 'no'}</td>"
        "</tr>"
    )

html_table = (
    "<h3>Data Validation Drift Summary</h3>"
    "<table>"
    "<thead><tr><th>feature</th><th>baseline mean</th><th>production mean</th>"
    "<th>mean shift</th><th>production missing</th><th>PSI</th><th>alert</th></tr></thead>"
    f"<tbody>{''.join(rows)}</tbody></table>"
)
html_path = ARTIFACT_DIR / "drift_summary.html"
html_path.write_text(
    "<!doctype html><meta charset='utf-8'><title>Data validation drift</title>"
    "<style>body{font-family:sans-serif}td,th{border:1px solid #ddd;padding:4px 8px}"
    "table{border-collapse:collapse}</style>"
    + html_table,
    encoding="utf-8",
)

try:
    from IPython.display import HTML, display
except ModuleNotFoundError:
    pass
else:
    display(HTML(html_table))

print(f"drift_json: {json_path}")
print(f"drift_csv: {csv_path}")
print(f"drift_html: {html_path}")
print(f"features_checked: {len(FEATURES)}")
print(f"alerts: {len(alerts)}")
for alert in alerts:
    print(
        f"- {alert['feature']}: psi={alert['psi']:.3f}, "
        f"missing={alert['missing']}, mean_shift={alert['mean_shift']:.3f}"
    )

ML_TASK_STATUS = "ok"


feature,baseline mean,production mean,mean shift,production missing,PSI,alert
support_load,0.010,0.425,0.415,14,0.328,yes
usage,-0.100,0.330,0.430,0,0.251,yes
nps,-0.002,-0.236,-0.233,19,0.084,yes
seat_ratio,0.034,-0.175,-0.209,0,0.081,no
tenure,-0.015,-0.113,-0.099,0,0.019,no


drift_json: artifacts/agent_rounds/20260615_round2/data_validation/data_validation_drift.json
drift_csv: artifacts/agent_rounds/20260615_round2/data_validation/feature_drift_summary.csv
drift_html: artifacts/agent_rounds/20260615_round2/data_validation/drift_summary.html
features_checked: 5
alerts: 3
- support_load: psi=0.328, missing=14, mean_shift=0.415
- nps: psi=0.084, missing=19, mean_shift=-0.233
- usage: psi=0.251, missing=0, mean_shift=0.430


In [2]:
from __future__ import annotations

import csv
import html
import json
from pathlib import Path


ARTIFACT_DIR = Path("artifacts/agent_rounds/20260615_round2/data_validation")
json_path = ARTIFACT_DIR / "data_validation_drift.json"
csv_path = ARTIFACT_DIR / "feature_drift_summary.csv"
html_path = ARTIFACT_DIR / "drift_summary.html"

for path in (json_path, csv_path, html_path):
    assert path.exists(), f"missing expected artifact: {path}"

payload = json.loads(json_path.read_text(encoding="utf-8"))
with csv_path.open(encoding="utf-8", newline="") as handle:
    rows = list(csv.DictReader(handle))

alerts = payload["alerts"]
top = max(rows, key=lambda row: float(row["psi"]))

print("artifact_manifest:")
for path in (json_path, csv_path, html_path):
    print(f"- {path} ({path.stat().st_size} bytes)")
print(f"json_features: {len(payload['features'])}")
print(f"json_alerts: {len(alerts)}")
print(f"csv_rows: {len(rows)}")
print(f"top_psi_feature: {top['feature']} psi={float(top['psi']):.3f}")
print("alert_features: " + ", ".join(alert["feature"] for alert in alerts))

preview_rows = []
for row in rows:
    preview_rows.append(
        "<tr>"
        f"<td>{html.escape(row['feature'])}</td>"
        f"<td>{html.escape(row['psi'])}</td>"
        f"<td>{html.escape(row['production_missing'])}</td>"
        f"<td>{html.escape(row['alert'])}</td>"
        "</tr>"
    )

preview = (
    "<h3>Artifact Inspection Preview</h3>"
    "<table><thead><tr><th>feature</th><th>PSI</th><th>missing</th><th>alert</th>"
    f"</tr></thead><tbody>{''.join(preview_rows)}</tbody></table>"
)

try:
    from IPython.display import HTML, display
except ModuleNotFoundError:
    pass
else:
    display(HTML(preview))

INSPECTION_STATUS = "ok"


artifact_manifest:
- artifacts/agent_rounds/20260615_round2/data_validation/data_validation_drift.json (2741 bytes)
- artifacts/agent_rounds/20260615_round2/data_validation/feature_drift_summary.csv (353 bytes)
- artifacts/agent_rounds/20260615_round2/data_validation/drift_summary.html (942 bytes)
json_features: 5
json_alerts: 3
csv_rows: 5
top_psi_feature: support_load psi=0.328
alert_features: support_load, nps, usage


feature,PSI,missing,alert
support_load,0.328086,14,True
usage,0.250880,0,True
nps,0.083955,19,True
seat_ratio,0.080652,0,False
tenure,0.018528,0,False


# Finding: data validation drift

Status: clean-run evidence available. `run-clean` completed with `status: ok`, `executed_cells: 2`, and `total_code_cells: 2`.

Evidence produced:

- `artifacts/agent_rounds/20260615_round2/data_validation/data_validation_drift.json` (2741 bytes)
- `artifacts/agent_rounds/20260615_round2/data_validation/feature_drift_summary.csv` (353 bytes)
- `artifacts/agent_rounds/20260615_round2/data_validation/drift_summary.html` (942 bytes)

Result: production data triggered 3 alerts across 5 checked features. `support_load` had the largest PSI (`0.328`) and 14 missing production values. `usage` also crossed the PSI threshold (`0.251`). `nps` had 19 missing production values.

Notebook inspection: `state --outputs summary` and `show-cell --outputs full` exposed both the text artifact manifest and readable HTML table previews, which was enough to follow the workflow without opening the raw notebook.
